# Chapter 11 &mdash; The Hill/Valley Plot, and Proofs of Consistency and Completeness

**Concept 8 of the Chapter 11 decomposition:** *The Hill/Valley Plot, and Proofs of Consistency and Completeness*

Plot the running $a$-minus-$b$ count; the curve's oscillation about the axis structures the proof.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-Hill-Valley-Plot/Concept-Hill-Valley-Plot.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


For a language defined by a **counting** condition, draw the running count as a curve.
For $L_{eq}$ = equal numbers of `a` and `b`, plot $\#a - \#b$ after each symbol:

* the curve **starts and ends at 0**;
* it goes **up** on `a` and **down** on `b`;
* it may wander **above** (a hill) and **below** (a valley) the axis freely.

The picture drives the proof. A string in $L_{eq}$ either stays on one side (so it
looks like $a\,x\,b$ or $b\,x\,a$ with $x\in L_{eq}$) or **touches zero in the
middle**, splitting it as $xy$ with both halves in $L_{eq}$.

Those two cases are exactly the productions, and the case analysis *is* the
completeness proof.

## 2. Definitions

### The CFG toolkit

A grammar is a dict; `language`, `nparses`, `parse_trees` and `leftmost` do the work.

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps

### The plot

In [ ]:
def plot(s):
    h, out = 0, []
    for ch in s:
        h += 1 if ch == 'a' else -1
        out.append(h)
    return out

def draw(s):
    ys = [0] + plot(s)
    lo, hi = min(ys), max(ys)
    for level in range(hi, lo - 1, -1):
        row = ''.join('*' if y == level else ('-' if level == 0 else ' ') for y in ys)
        print("  %3d |%s" % (level, row))
    print("       " + ' ' + s)

### The grammar the picture suggests

In [ ]:
Leq = mkg({'S': ["", "aSb", "bSa", "SS"]})
def equal_ab(s): return s.count('a') == s.count('b')

## 3. Tests

A hill, a valley, and a curve that touches zero in the middle.

In [ ]:
for s in ['aabb', 'bbaa', 'abba', 'abab']:
    print("s =", s, " ends at", plot(s)[-1])
    draw(s)
    print()

The grammar generates exactly the equal-count strings.

In [ ]:
from itertools import product
L = set(language(Leq, 8))
want = {''.join(p) for k in range(0, 9, 2) for p in product('ab', repeat=k)
        if equal_ab(''.join(p))}
print("generated %d, intended %d" % (len(L), len(want)))
print("missing :", sorted(want - L)[:5], " extra :", sorted(L - want)[:5])
assert L == want

**Consistency** &mdash; each production preserves the count difference.

In [ ]:
print("S -> ''    : 0 - 0 = 0            ok")
print("S -> aSb   : (1 + k) - (1 + k) = 0 ok")
print("S -> bSa   : symmetric             ok")
print("S -> SS    : 0 + 0 = 0             ok")
assert all(equal_ab(w) for w in language(Leq, 8))
print("\nverified on every generated string up to length 8")

**Completeness** &mdash; the case analysis the picture makes obvious.

In [ ]:
def split_case(s):
    ys = plot(s)
    zeros = [i for i, y in enumerate(ys[:-1]) if y == 0]
    if zeros: return ('join', s[:zeros[0]+1], s[zeros[0]+1:])
    return ('wrap', s[0], s[1:-1], s[-1])

for s in ['abab', 'aabb', 'bbaa', 'abba', 'aababb']:
    print("%-8s ->" % s, split_case(s))
print("\nTouches zero in the middle  -> the SS rule.")
print("Never touches zero in between -> aSb or bSa, and the inside is shorter.")

The induction bottoms out because both cases shorten the string.

In [ ]:
def check(s):
    if s == '': return True
    kind = split_case(s)
    if kind[0] == 'join':
        return check(kind[1]) and check(kind[2])
    return kind[1] != kind[3] and check(kind[2])
assert all(check(w) for w in want)
print("the recursive case analysis terminates and succeeds on all %d strings"
      % len(want))

## 4. Exercises


1. Draw the plot for `aabbab`. Which rule applies at the top level?
2. Why does "never touches zero in between" force the first and last symbols to differ?
3. Adapt the plot to $\#a = 2\#b$. What is the grammar?

In [ ]:
# Your work for the exercises above.